# Image Prompting & Visual Reasoning — Getting VLMs to See and Think

> **Orbit 5 (Multimodal)** · Notebook 5 of 22

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/cyruslayo/castalia/blob/main/multimodal/05_image_prompting_and_visual_reasoning.ipynb)

## Learning Objectives

By the end of this notebook you will be able to:

1. **Explain** how Vision-Language Models process images via patch tokenization and linear projection into a shared embedding space.
2. **Apply** five distinct visual prompting strategies — direct question, instructed analysis, spatial reference, chain-of-thought, and structured extraction — and know when each is most effective.
3. **Implement** chain-of-thought visual reasoning to boost performance on counting, spatial, and text-reading tasks.
4. **Diagnose** common VLM failure modes including object hallucination, counting errors, spatial confusion, and text misreading.
5. **Measure** VLM accuracy on visual tasks using a reproducible mini-benchmark with synthetic test images.

## How VLMs Process Images — From Patches to Understanding

If you completed **foundations/05** on tokenization, you already know that language models convert text into tokens — discrete integer IDs that index into an embedding table. Vision-Language Models do something remarkably similar with images, and understanding this pipeline is the key to effective visual prompt engineering.

### The Vision Encoder Pipeline

The journey from pixels to understanding follows four stages:

1. **Patch extraction.** The raw image is sliced into a grid of fixed-size patches. A standard ViT-based encoder splits a 224×224 image into 14×14-pixel patches, yielding **256 patches** (a 16×16 grid). Each patch is flattened into a pixel vector.
2. **Linear projection.** Each flattened patch is passed through a learned linear layer that maps the raw pixel values into the model's hidden dimension — exactly like a token embedding layer. The result: 256 *visual tokens*, each living in the same vector space as text tokens.
3. **Positional encoding.** Spatial position embeddings are added so the model knows which patch came from which region of the image. Without these, the model would see an unordered bag of patches.
4. **Cross-modal attention.** The visual tokens are concatenated with (or cross-attended to) the text tokens from your prompt. This is where "understanding" happens: attention heads learn to bind the word *"red"* to the patch containing red pixels, or the phrase *"bottom-left"* to the corresponding spatial position.

Qwen2.5-VL uses **dynamic resolution** — unlike fixed-resolution models, it adapts its patch grid to the actual input size. A small 224×224 image might produce ~256 tokens, while a large 1344×896 image could produce ~1,500+ tokens. This flexibility means image size directly impacts inference cost and context budget.

### Why This Matters for Prompt Engineering

Your text prompt steers attention allocation across those visual tokens. A vague prompt like *"describe this"* lets the model attend everywhere equally — often producing generic, unfocused output. A precise prompt like *"what color is the object in the top-left corner?"* concentrates attention on a specific spatial region, dramatically improving accuracy. The five strategies we explore below are fundamentally techniques for **controlling where and how the model attends** to its visual tokens.

In [ ]:
!pip install -q "transformers>=4.57.0" accelerate bitsandbytes torch pillow qwen-vl-utils requests pandas

In [ ]:
import torch, json, time, requests, random, textwrap
from io import BytesIO
from PIL import Image, ImageDraw, ImageFont
from transformers import Qwen2_5_VLForConditionalGeneration, AutoProcessor, BitsAndBytesConfig
import pandas as pd

bnb = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_compute_dtype=torch.float16)
MODEL_ID = "Qwen/Qwen2.5-VL-7B-Instruct"
model = Qwen2_5_VLForConditionalGeneration.from_pretrained(
    MODEL_ID, quantization_config=bnb, device_map="auto"
)
processor = AutoProcessor.from_pretrained(MODEL_ID)
print(f"✓ {MODEL_ID} loaded  |  VRAM: {torch.cuda.memory_allocated()/1024**3:.1f} GB")

### Helper: `ask_vlm`

We wrap the full generate-and-decode loop in a single function so every experiment below is a clean one-liner. The function accepts a PIL image and a text prompt, builds the multi-modal message list expected by Qwen2.5-VL, and returns the decoded response.

In [ ]:
def ask_vlm(image, prompt, max_tokens=512):
    """Send an image + text prompt to the VLM and return the text response."""
    messages = [{"role": "user", "content": [
        {"type": "image", "image": image},
        {"type": "text", "text": prompt},
    ]}]
    text = processor.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = processor(
        text=[text], images=[image], return_tensors="pt", padding=True
    ).to(model.device)
    with torch.no_grad():
        ids = model.generate(**inputs, max_new_tokens=max_tokens)
    return processor.batch_decode(
        ids[:, inputs.input_ids.shape[1]:], skip_special_tokens=True
    )[0]

## Building Synthetic Test Images

We create our own test images with PIL rather than relying on external URLs. This gives us **ground-truth labels** — we know exactly what objects exist, what colors they are, and where they sit. That makes it possible to objectively score VLM responses later in the notebook.

In [ ]:
def make_test_scene():
    """Generate a synthetic scene with three labeled geometric shapes."""
    img = Image.new("RGB", (640, 480), "white")
    draw = ImageDraw.Draw(img)
    # Red rectangle — left region
    draw.rectangle([50, 100, 200, 300], fill="red", outline="black", width=2)
    draw.text((100, 310), "Box A", fill="black")
    # Blue circle — center region
    draw.ellipse([260, 120, 400, 260], fill="blue", outline="black", width=2)
    draw.text((305, 270), "Circle B", fill="black")
    # Green triangle — right region
    draw.polygon([(500, 300), (450, 150), (550, 150)], fill="green", outline="black")
    draw.text((470, 310), "Triangle C", fill="black")
    # Metadata text
    draw.text((20, 20), "Test Scene v1.0", fill="gray")
    draw.text((20, 440), "Objects: 3  |  Colors: R, G, B", fill="gray")
    return img

scene = make_test_scene()
scene

## The Visual Prompting Toolkit

Just as text prompts range from lazy (*"summarize this"*) to precise (*"extract the three key arguments and list counter-evidence for each"*), visual prompts exist on a spectrum of specificity. We will test five strategies on our synthetic scene:

| # | Strategy | Core Idea |
|---|----------|----------|
| 1 | **Direct Question** | Open-ended; lets the model decide what to focus on |
| 2 | **Instructed Analysis** | Tells the model *what* to report for each object |
| 3 | **Spatial Reference** | Directs attention to a specific image region |
| 4 | **Chain-of-Thought** | Breaks the task into numbered reasoning steps |
| 5 | **Structured Extraction** | Demands output in a machine-parseable format (JSON) |

Think of it like asking a human to look at a photograph. *"What's this?"* gets a one-sentence answer. *"Count the objects left-to-right and describe each one's shape, color, and position"* produces a detailed inventory. The model is no different — **specificity in the prompt drives specificity in the output**.

In [ ]:
print("═══ Strategy 1: Direct Question ═══")
response = ask_vlm(scene, "What is in this image?")
print(response)

In [ ]:
print("═══ Strategy 2: Instructed Analysis ═══")
response = ask_vlm(
    scene,
    "List every object in this image. For each, state its shape, color, "
    "and approximate position (left/center/right)."
)
print(response)

In [ ]:
print("═══ Strategy 3: Spatial Reference ═══")
response = ask_vlm(
    scene,
    "Describe the object in the left third of the image. "
    "What color is it? What shape? What label appears below it?"
)
print(response)

In [ ]:
print("═══ Strategy 4: Chain-of-Thought ═══")
prompt = """Look at this image step by step:
1. First, list every distinct object you can see.
2. For each object, state its color and shape.
3. Describe the spatial layout (what is left of what).
4. Read any text visible in the image.
5. Finally, summarize the scene in one sentence."""
response = ask_vlm(scene, prompt, max_tokens=800)
print(response)

In [ ]:
print("═══ Strategy 5: Structured Extraction ═══")
prompt = """Extract information from this image as JSON:
{
  "objects": [{"name": "...", "color": "...", "shape": "...", "position": "..."}],
  "text_content": ["..."],
  "object_count": N
}
Output ONLY valid JSON."""
response = ask_vlm(scene, prompt, max_tokens=600)
print(response)

# Attempt to parse — a good test of extraction quality
try:
    parsed = json.loads(response)
    print(f"\n✓ Valid JSON with {len(parsed.get('objects', []))} objects")
except json.JSONDecodeError as e:
    print(f"\n✗ JSON parse error: {e}")

### Strategy Comparison

| Strategy | Specificity | Structure | Token Cost | Best For |
|----------|:-----------:|:---------:|:----------:|----------|
| Direct Question | Low | None | ~50 | Quick sanity checks |
| Instructed Analysis | Medium | Implicit | ~150 | Inventories, scene descriptions |
| Spatial Reference | High (region) | Implicit | ~100 | Localized detail extraction |
| Chain-of-Thought | High | Sequential | ~300 | Complex reasoning, counting |
| Structured Extraction | High | Explicit (JSON) | ~200 | Downstream pipelines, databases |

> **Rule of thumb:** Start with Strategy 2 for general tasks. Upgrade to Strategy 4 when accuracy matters. Use Strategy 5 when a downstream system will consume the output.

## Visual Reasoning Tasks

Description is the *easy* task for VLMs — it is what they were trained on most heavily (image-caption pairs). The harder, more revealing tasks are those that require genuine **visual reasoning**:

- **Counting**: How many objects of type X? This requires individuating objects — distinguishing "three overlapping circles" from "one blob."
- **Spatial reasoning**: Is object A above or below object B? Left or right? These require binding identity to position.
- **Text reading (OCR)**: Extract exact strings from rendered text. The model must decode glyph shapes from pixel patches.
- **Comparative analysis**: What changed between two versions of an image? This demands aligned attention across two inputs.

These tasks expose the real limits of current VLMs. Counting is unreliable above ~8-10 objects; spatial reasoning degrades when objects are close together; text reading fails on small or stylized fonts. Understanding these limits tells you when to trust the model and when to add verification.

In [ ]:
def make_counting_image(n_circles=7):
    """Generate an image with n_circles colored circles at pseudo-random positions."""
    rng = random.Random(42)
    img = Image.new("RGB", (400, 400), "white")
    draw = ImageDraw.Draw(img)
    colors = ["red", "blue", "green", "orange", "purple"]
    for i in range(n_circles):
        x = rng.randint(30, 370)
        y = rng.randint(30, 370)
        r = 15
        draw.ellipse([x - r, y - r, x + r, y + r],
                     fill=colors[i % len(colors)], outline="black")
    return img, n_circles

In [ ]:
count_img, true_count = make_counting_image(7)
response = ask_vlm(
    count_img,
    "Count the total number of circles in this image. Reply with just the number."
)
print(f"True count: {true_count}")
print(f"Model says: {response}")
count_img

### Spatial Relationship Probing

In [ ]:
spatial_questions = [
    "Is the red rectangle to the left or right of the blue circle?",
    "Which object is closest to the top of the image?",
    "What color is the shape furthest to the right?",
]
for q in spatial_questions:
    ans = ask_vlm(scene, q, max_tokens=100)
    print(f"Q: {q}")
    print(f"A: {ans}\n")

In [ ]:
def make_text_image():
    """Generate an image containing structured text lines."""
    img = Image.new("RGB", (500, 200), "white")
    draw = ImageDraw.Draw(img)
    lines = ["Invoice #2024-0831", "Total: $1,247.50", "Due: March 15, 2025"]
    y = 20
    for line in lines:
        draw.text((30, y), line, fill="black")
        y += 50
    return img, lines

text_img, ground_truth = make_text_image()
response = ask_vlm(text_img, "Read all text in this image. List each line exactly as written.")
print("Ground truth:", ground_truth)
print("Model output:", response)
text_img

## Common VLM Failure Modes

Every tool has characteristic failure modes. Knowing them lets you design prompts that avoid the worst pitfalls and add guardrails where the model is weakest.

| Failure Mode | What Happens | Root Cause | Mitigation |
|---|---|---|---|
| **Object hallucination** | Model reports objects that do not exist | Training-data priors; the model "expects" certain objects in certain scenes | Ask the model to verify: *"Are you sure X is present? Look again."* |
| **Counting errors** | Miscounts, especially above 8-10 items | Attention cannot individuate densely packed patches | Use CoT; ask the model to label each object before summing |
| **Spatial confusion** | Swaps left/right, above/below | Position embeddings lose precision at high token counts | Refer to specific regions; crop the image to isolate areas |
| **Text misreading** | Character substitutions, missing digits | Small text becomes blurry after patch downsampling | Increase image resolution; zoom into text regions |
| **Color confusion** | Mislabels colors, especially on small objects | Color information is averaged across a patch | Ensure objects span multiple patches; avoid anti-aliased edges |

The unifying theme is **patch resolution**. When an object is too small, it occupies a fraction of a single patch — its color, shape, and identity are mixed with background pixels during projection. This is why zooming in, cropping, or using higher-resolution inputs can dramatically improve accuracy.

In [ ]:
print("── Failure Mode: Counting with Many Objects ──")
hard_img, true_n = make_counting_image(15)
response = ask_vlm(
    hard_img,
    "How many circles are in this image? Count carefully."
)
print(f"True: {true_n}, Model: {response}")
print()

# Retry with chain-of-thought mitigation
print("── Retry with CoT Mitigation ──")
response_cot = ask_vlm(
    hard_img,
    "Count the circles step by step. First describe where each circle "
    "is located, then give the total count.",
    max_tokens=800,
)
print(f"With CoT: {response_cot}")
hard_img

## Measuring VLM Accuracy

You cannot improve what you do not measure. In text-only NLP, benchmarks like MMLU and HellaSwag provide standardized evaluation. For vision tasks, we build our own task-specific benchmarks — and synthetic images make this trivially reproducible.

Our mini-benchmark below tests three capabilities: **counting**, **spatial reasoning**, and **text reading**. For each task we know the ground-truth answer, so we can check whether the model's response contains the correct value. This is a simple *substring-match* metric — production systems would use more sophisticated scoring — but it reveals meaningful patterns.

Key insight: run the benchmark **before and after** changing your prompting strategy. The delta is what tells you whether a new strategy actually helps or just *feels* better.

In [ ]:
def run_mini_benchmark(model_fn):
    """Run a small benchmark across counting, spatial, and text tasks."""
    tasks = [
        {
            "type": "count",
            "image_fn": lambda: make_counting_image(3),
            "question": "How many circles? Reply with just the number.",
            "answer": "3",
        },
        {
            "type": "count",
            "image_fn": lambda: make_counting_image(5),
            "question": "How many circles? Reply with just the number.",
            "answer": "5",
        },
        {
            "type": "count",
            "image_fn": lambda: make_counting_image(10),
            "question": "How many circles? Reply with just the number.",
            "answer": "10",
        },
        {
            "type": "spatial",
            "image_fn": lambda: (make_test_scene(), None),
            "question": "What color is the object on the far left?",
            "answer": "red",
        },
        {
            "type": "text",
            "image_fn": lambda: make_text_image(),
            "question": "What is the invoice number?",
            "answer": "2024-0831",
        },
    ]
    results = []
    for t in tasks:
        img, _ = t["image_fn"]()
        resp = model_fn(img, t["question"], max_tokens=50)
        correct = t["answer"].lower() in resp.lower()
        results.append({
            "type": t["type"],
            "expected": t["answer"],
            "got": resp.strip()[:60],
            "correct": correct,
        })
    return pd.DataFrame(results)

results = run_mini_benchmark(ask_vlm)
print(results.to_string(index=False))
print(f"\nAccuracy: {results['correct'].mean():.0%}")

### Re-running with Chain-of-Thought

Let us wrap `ask_vlm` with a CoT prefix and re-run the same benchmark to see if step-by-step reasoning helps.

In [ ]:
def ask_vlm_cot(image, prompt, max_tokens=512):
    """Wrapper that prepends a chain-of-thought instruction to every prompt."""
    cot_prefix = "Think step by step before answering. "
    return ask_vlm(image, cot_prefix + prompt, max_tokens=max_tokens)

results_cot = run_mini_benchmark(ask_vlm_cot)
print(results_cot.to_string(index=False))
print(f"\nCoT Accuracy: {results_cot['correct'].mean():.0%}")

# Side-by-side comparison
comparison = results[["type", "expected"]].copy()
comparison["direct"] = results["correct"]
comparison["cot"] = results_cot["correct"]
print("\n── Direct vs CoT ──")
print(comparison.to_string(index=False))

### Bonus: Effect of Image Resolution

Since Qwen2.5-VL uses dynamic resolution, we can test whether resizing our synthetic image changes accuracy. Larger images produce more visual tokens, giving the model finer-grained patch information at the cost of higher latency.

In [ ]:
print("── Resolution Experiment: Counting 10 circles ──")
base_img, _ = make_counting_image(10)

for size in [200, 400, 800]:
    resized = base_img.resize((size, size), Image.LANCZOS)
    t0 = time.time()
    resp = ask_vlm(resized, "How many circles? Reply with just the number.", max_tokens=20)
    elapsed = time.time() - t0
    print(f"  {size}×{size}: model says {resp.strip():>4s}  ({elapsed:.1f}s)")

## Exercises

**Exercise 1 — Real-Image Strategy Sweep.** Download a real photograph from a URL (e.g., a Wikimedia Commons image). Apply all five prompting strategies from this notebook. Compare the outputs: which strategy produces the most useful response? Does the ranking match what you observed on synthetic images?

**Exercise 2 — Counting Stress Test.** Modify `make_counting_image` to generate scenes with overlapping circles of varying sizes (radii between 5 and 30 pixels). Run the counting test for n = 5, 10, 15, 20, 25, 30. Plot accuracy vs. object count. At what count does accuracy drop below 50%?

**Exercise 3 — Visual QA Benchmark.** Design a complex scene (multiple shapes, colors, text labels, overlapping objects). Write 10 factual questions with ground-truth answers. Run each question with and without chain-of-thought. Report accuracy for both conditions and compute the improvement delta.

In [ ]:
# Exercise 1 starter: Try with a real image
# url = "https://upload.wikimedia.org/wikipedia/commons/thumb/a/a7/Camponotus_flavomarginatus_ant.jpg/640px-Camponotus_flavomarginatus_ant.jpg"
# resp = requests.get(url)
# real_img = Image.open(BytesIO(resp.content)).convert("RGB")
#
# strategies = {
#     "direct": "What is in this image?",
#     "instructed": "List every object. For each, state shape, color, and position.",
#     "spatial": "Describe the object in the center of the image in detail.",
#     "cot": "Step by step: 1) List objects. 2) Describe colors. 3) Describe layout. 4) Summarize.",
#     "json": 'Extract as JSON: {"objects": [{"name": ..., "color": ...}], "count": N}',
# }
#
# for name, prompt in strategies.items():
#     print(f"--- {name} ---")
#     print(ask_vlm(real_img, prompt))
#     print()

In [ ]:
# Exercise 2 starter: Counting stress test
# import matplotlib.pyplot as plt
#
# counts = [5, 10, 15, 20, 25, 30]
# accuracies = []
# for n in counts:
#     img, true = make_counting_image(n)
#     resp = ask_vlm(img, "How many circles? Reply with just the number.", max_tokens=20)
#     try:
#         predicted = int("".join(c for c in resp if c.isdigit())[:3])
#     except ValueError:
#         predicted = -1
#     accuracies.append(1.0 if predicted == true else 0.0)
#     print(f"n={n:2d}  predicted={predicted}  correct={predicted == true}")
#
# plt.plot(counts, accuracies, "o-")
# plt.xlabel("True count"); plt.ylabel("Accuracy")
# plt.title("Counting accuracy vs object count"); plt.grid(True); plt.show()

## Key Takeaways

1. **VLMs tokenize images into patches** — your text prompt steers attention over those visual tokens, just as it steers attention over text tokens.
2. **Five prompting strategies** span a spectrum from low-effort (direct question) to high-structure (JSON extraction). Match the strategy to the task.
3. **Chain-of-thought prompting** significantly improves performance on counting, spatial reasoning, and multi-step analysis by forcing the model to show intermediate work.
4. **Always benchmark.** Synthetic test images with known ground truth let you measure exact accuracy and detect regressions when you change prompts or models.
5. **Know your failure modes.** Object hallucination, counting errors, and spatial confusion are the big three. Each has specific mitigations rooted in how patch tokenization works.

## References

1. **Qwen2.5-VL Technical Report** (2024) — Bai et al. Architecture details for dynamic-resolution vision encoding.
2. **"Visual Instruction Tuning"** — Liu, H. et al. (LLaVA, NeurIPS 2023). Foundational work on visual instruction following.
3. **"Chain-of-Thought Prompting Elicits Reasoning in Large Language Models"** — Wei, J. et al. (NeurIPS 2022). The CoT technique applied here to visual tasks.
4. **Castalia foundations/05 — Tokenization and Embedding.** Background on how tokenization works for text, which parallels patch tokenization for images.

---

[← 04 Building a Multimodal Benchmark Harness](04_building_a_multimodal_benchmark_harness.ipynb) | [06 OCR, Layout & Table Extraction →](06_ocr_layout_and_table_extraction.ipynb)